# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print("Name:", getattr(metadata, 'name', ''))
print("Description:", getattr(metadata, 'description', ''))

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

In [ ]:
from pprint import pprint

# List available record sets
print("Available Record Sets:")
record_sets = list(dataset.list_record_sets())  # returns list of (@id, label) tuples
for rec in record_sets:
    pprint({"@id": rec[0], "label": rec[1]})

# Choose the first record set for further exploration
if record_sets:
    selected_record_set_id = record_sets[0][0]
    print("\nFields in selected record set (by @id and label):")
    fields = list(dataset.list_fields(record_set=selected_record_set_id))  # returns (@id, label)
    for field in fields:
        pprint({"@id": field[0], "label": field[1]})
else:
    print("No record sets found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set (by their @id)
from collections import OrderedDict

dataframes = OrderedDict()
print("Loading data for all record sets found:")
for recset_id, recset_label in record_sets:
    records = list(dataset.records(record_set=recset_id))
    df = pd.DataFrame(records)
    dataframes[recset_id] = df
    print(f"  - {recset_id}: {df.shape[0]} records, Columns: {list(df.columns)[:6]}{'...' if df.shape[1] > 6 else ''}")

# Preview the first dataframe (from the first record set)
if record_sets:
    example_rs_id = record_sets[0][0]
    print(f"\nFirst 5 rows of record set '{example_rs_id}':")
    display_cols = dataframes[example_rs_id].columns.tolist()[:10]  # prevent wide output
    display(dataframes[example_rs_id][display_cols].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes using field `@id`s.

In [ ]:
# Automatically select a numeric field (e.g., coverage, peptide_count, MW, or abundance)
# We search for a numeric column. You may override numeric_field_id below with a specific @id from overview.
df = dataframes[example_rs_id]
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

if not numeric_field_id:
    print("No automatic numeric field detected. Please set 'numeric_field_id' to a column name from previous cell.")
else:
    print(f"Using numeric field: {numeric_field_id}")
    threshold = df[numeric_field_id].quantile(0.75)
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.3f} (upper quartile): {len(filtered_df)} rows.")
    # Normalize
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nFirst 5 filtered and normalized records:")
    display(filtered_df[[numeric_field_id, col_norm]].head())
    # Try to detect a categorical/groupable field
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and df[col].dtype == object and df[col].nunique() < 10:
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped data by '{group_field_id}' (mean of {numeric_field_id}):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id} (Filtered)")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated how to explore the FAIR² dataset on human protein mass spectrometry using the `mlcroissant` library.

- Dataset metadata and structure are introspected via Croissant schema.
- Record sets and fields are referenced using their canonical `@id`s.
- Tabular data is extracted into Pandas for analysis, with example transformation and visualizations provided.

Explore further by adjusting field `@id`s, using additional columns, or applying domain-specific bioinformatics analysis.